In [ ]:
import os, json, nilearn
import pandas as pd
import numpy as np

In [ ]:
times = []
folder = "/mnt/DATA/LEMON_Giulio"
for file in os.listdir(folder):
    if file.endswith("json") and file.startswith("sub"):
        with open(os.path.join(folder, file)) as fp:
            meta = json.load(fp)
        subject, session, protocol, direction, run, tail = file.split("_")
        times.append([subject, session, run, direction, meta["EchoTime"], meta["RepetitionTime"]])
times_df = pd.DataFrame(times, columns=["subject", "session", "run", "direction", "EchoTime", "RepetitionTime"])
        

In [ ]:
times_df[times_df.session == "ses-01"].describe()

In [ ]:
np.sum(np.isclose(times_df[times_df.session == "ses-02"].EchoTime, 0.03))

In [ ]:
good_sub = times_df[times_df.session == "ses-02"][np.isclose(times_df[times_df.session == "ses-02"].EchoTime, 0.03)].subject
good_session = times_df[times_df.subject.isin(good_sub)]

In [ ]:
good_session.sort_values(["subject","session","run","direction"], inplace=True)

In [ ]:
protocol = "task-rest"
tail = "bold."
with open(os.path.join(folder, "selected.txt"), "w") as fp:
    for row in good_session[good_session.direction == "acq-AP"].sort_values(["subject","session","run","direction"]).iterrows():
        pieces = list(row[1].iloc[:-2])
        pieces.insert(2, protocol)
        pieces[3], pieces[4] = pieces[4], pieces[3]
        pieces.append(tail)
        base_name = "_".join(pieces)
        for ext in ["json","nii.gz"]:
            assert os.path.isfile(os.path.join(folder, base_name+ext)),os.path.join(folder, base_name+ext)
            print(os.path.join(folder, base_name+ext), file=fp)


In [ ]:
_task-rest_acq-AP_run-01_bold.json/_acq-mp2rage_T1w.nii.gz

In [ ]:
from nilearn import datasets
from nilearn.maskers import NiftiLabelsMasker
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

dataset = datasets.fetch_atlas_aal("SPM12")
atlas_filename = dataset.maps
labels = dataset.labels
masker = NiftiLabelsMasker(labels_img=atlas_filename, standardize=True)
time_series = masker.fit_transform("/mnt/DATA/LEMON_Giulio/SELECTED/dswausub-010005_ses-01_task-rest_acq-AP_run-01_bold.nii")
cov = np.cov(time_series.T)
print(time_series.shape, cov.shape)
plt.imshow(cov)

In [ ]:
TR = 1.4
f_sample = 1/TR
sos = signal.butter(4, [0.009,0.08], 'bp', False, 'sos', f_sample)
filtered = signal.sosfiltfilt(sos, time_series[:,:], axis=0)
plt.plot(1.4*np.arange(657),time_series[:,45])
plt.plot(1.4*np.arange(657),filtered[:,45])
plt.plot(2*np.arange(460),signal.resample(filtered[:,45], 460))
plt.plot(5.55*np.arange(166),signal.resample(filtered[:,45], 166))


In [ ]:
plt.plot(np.arange(100)/(1.4*657),np.fft.fft(time_series[:,0])[:100])

In [ ]:
cov2 = np.cov(filtered.T)
print(filtered.shape, cov2.shape)
plt.imshow(cov2[:-26,:-26])

In [ ]:
1/0.16

In [ ]:
1.4*657/60, 657*1.4/6.25*1.125

In [ ]:
6.25/1.125

In [ ]:
sessions = ["ses-01_task-rest_acq-AP_run-01", "ses-02_task-rest_acq-AP_run-01", "ses-02_task-rest_acq-AP_run-02"]
from glob import glob
from scipy.io import savemat
import numpy as np
from tqdm import tqdm

for i, session in tqdm(enumerate(sessions), desc = "Session", total=len(sessions)):
    files = sorted(glob(f"/mnt/DATA/LEMON_Giulio/SELECTED/dswausub-*_{session}_bold.nii"))
    newmat = np.empty([657, 90, 14])
    for j, file in tqdm(enumerate(files), desc = "Subject", total=len(files)):
        time_series = masker.fit_transform(file)
        newmat[:,:,j]=time_series[:,:90]
    savemat(f"/mnt/DATA/LEMON_Giulio/processed/lemon_aal_strin_90_ses-{i:02}.mat", {"subj_tcs":newmat})

In [ ]:
657**0.333333